# Knowledge Retrieval (RAG)

Retrieval-Augmented Generation (RAG) grounds LLM responses in external, up-to-date knowledge: retrieve relevant document chunks -> augment the prompt -> generate a grounded answer.

## Implementation with Flyte v2 + the Agent harness

The original LangChain + LangGraph version wired a fixed `StateGraph`: **always** retrieve, then **always** generate. This notebook refactors it into a Flyte v2 `Agent` with a single `retrieve_documents` tool. Retrieval becomes **agentic**: the model decides _when_ it needs to look something up versus answering directly.

To keep the example dependency-light and reliable on the local devbox, retrieval here is a **lightweight keyword scorer** (pure Python, no embedding model). The agentic RAG *pattern* is identical to a production system; only the retrieval backend differs — swap in a vector DB (ChromaDB, Weaviate, pgvector) for semantic search when you need it.

#### LangChain / LangGraph vs Flyte v2 + Agent harness

| Aspect | LangChain / LangGraph | Flyte v2 + `Agent` harness |
|--------|----------------------|----------------------------|
| **Pipeline shape** | Fixed `retrieve -> generate` edges | Agent decides when to call `retrieve_documents` |
| **Re-retrieval** | Needs an explicit cycle node | Agent can retrieve again within `max_turns` |
| **Retrieval backend** | Vector store (embeddings) | Lightweight keyword scoring (pure Python) — swap a vector DB in for production |
| **Retrieval step** | A graph node | An in-process `@flyte.trace` tool, shown as a nested action |
| **LLM client** | `ChatOpenAI` | Harness' litellm callback |
| **Output** | `TypedDict` state | Typed `AgentResult` |
| **Secrets** | `.env` / `os.environ` | `flyte.Secret` injected by cluster |

> **🧭 When to use this pattern — and how Flyte helps**
>
> Use retrieval when answers must be grounded in external, current, or proprietary knowledge rather than the model's parametric memory. In Flyte the retriever is an `@env.task` tool — cached with `cache="auto"` to skip re-embedding — and the agent decides when to call it, so trivial queries skip retrieval entirely.

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' litellm

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [2]:
!flyte start devbox

  Waiting for flyte cluster to be ready ━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:02:005m 83% 0:02:00
╭──────────────────────────────── Flyte Devbox ────────────────────────────────╮
│ Flyte devbox cluster is ready!                                               │
│                                                                              │
│   🚀 UI:             ]8;id=13558273;http://localhost:30080/v2\http://localhost:30080/v2]8;;\                               │
│   🐳 Image Registry: localhost:30000                                         │
╰──────────────────────────────────────────────────────────────────────────────╯


### 2. Store your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-proj-...

### 3. Import dependencies and configure the TaskEnvironment

In [ ]:
import os
from datetime import timedelta

import flyte
from flyte.ai.agents import Agent, AgentResult

flyte.init_from_config()

# No ML/embedding deps: retrieval is pure Python, so the image is just litellm.
_image = (
    flyte.Image.from_debian_base(name="rag-agent-lite", python_version=(3, 12))
    .with_pip_packages("litellm")
)

rag_env = flyte.TaskEnvironment(
    name="rag_pipeline",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="4Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
)

### 4. Define the retrieval tool

In the Agent harness, retrieval is **one tool the model calls on demand**; generation is the agent's own job. `retrieve_documents` is an in-process `@flyte.trace` (so it shows up as a nested, traced sub-action in the UI without spawning a separate pod) that scores knowledge-base chunks against the question by keyword overlap and returns the best passages.

This keyword scorer is deliberately simple — it needs no embedding model, no vector DB, and no model download, so it runs instantly and reliably on the devbox. For real semantic search, replace the body with a vector-store query (the agent, tool interface, and grounding pattern stay exactly the same).

In [2]:
KNOWLEDGE_BASE = """
Flyte is a cloud-native workflow orchestration platform designed for machine learning and data pipelines.
It was originally developed at Lyft and open-sourced in 2021. Flyte provides a type-safe, reproducible,
and scalable way to define and execute workflows using Python. Tasks in Flyte are containerized functions
that run on Kubernetes. Each task can be assigned specific compute resources like CPU, memory, and GPU.
Flyte supports caching of task outputs, which means if a task has already run with the same inputs, it
returns the cached result instead of recomputing. The Flyte v2 SDK uses a TaskEnvironment to group tasks
that share the same container image and resource configuration. Flyte also supports reusable containers
through ReusePolicy, which keeps warm containers around to reduce cold-start latency. Secrets are managed
securely through the flyte.Secret API, which injects credentials as environment variables at task execution
time without storing them in code. The Flyte UI provides real-time visibility into workflow execution,
including task inputs, outputs, logs, and custom HTML reports.
"""

_STOPWORDS = {
    "the", "a", "an", "is", "are", "was", "were", "be", "to", "of", "and", "or", "in",
    "on", "for", "with", "how", "what", "where", "which", "does", "do", "it", "its",
    "that", "this", "as", "at", "by", "from", "can", "you", "your",
}


def _chunk_text(text: str, chunk_size: int = 350, overlap: int = 40) -> list[str]:
    """Split text into overlapping chunks on sentence boundaries."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        boundary = text.rfind(".", start, end)
        if boundary > start + chunk_size // 2:
            end = boundary + 1
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap
    return chunks


@flyte.trace
async def retrieve_documents(question: str, top_k: int = 3) -> str:
    """Search the knowledge base for passages relevant to a question.

    Call this whenever the user asks about Flyte and you need specific facts to
    answer accurately. Returns the most relevant passages as text.

    Args:
        question: The information need to search for.
        top_k: How many passages to return.
    """
    import re

    chunks = _chunk_text(KNOWLEDGE_BASE)
    q_terms = set(re.findall(r"\w+", question.lower())) - _STOPWORDS

    def score(chunk: str) -> int:
        words = re.findall(r"\w+", chunk.lower())
        return sum(1 for w in words if w in q_terms)

    ranked = sorted(chunks, key=score, reverse=True)
    hits = [c for c in ranked if score(c) > 0][:top_k] or ranked[:top_k]
    return "\n\n".join(f"[passage {i}] {p}" for i, p in enumerate(hits))

### 5. Build the RAG agent

#### From fixed pipeline to agentic retrieval

The agent  **decides** whether to retrieve. It also closes the loop the `StateGraph` couldn't without an explicit cycle: if the first passages don't answer the question, the model can call `retrieve_documents` again before answering.

In [3]:
rag_agent = Agent(
    name="rag-assistant",
    model="claude-haiku-4-5",
    instructions=(
        "You answer questions about Flyte. When a question needs specific facts, call "
        "retrieve_documents to fetch relevant passages, then answer ONLY from what you "
        "retrieved and keep it to about 3 sentences. For greetings or general chit-chat, "
        "answer directly without retrieving. If the passages don't contain the answer, "
        "say you don't know."
    ),
    tools=[retrieve_documents],
    max_turns=4,
)


@rag_env.task(cache=flyte.Cache(behavior="disable"))
async def rag_pipeline(question: str) -> str:
    """Agentic RAG: the agent decides whether to retrieve before answering."""
    result: AgentResult = await rag_agent.run.aio(question)
    if result.error:
        raise RuntimeError(result.error)
    return result.summary

### 7. Run locally

In [4]:
QUESTIONS = [
    "What is Flyte and where was it originally developed?",
    "How does Flyte handle secrets?",
    "What is a TaskEnvironment in Flyte v2?",
]

for question in QUESTIONS:
    run = flyte.run(rag_pipeline, question=question)
    run.wait()
    answer = run.outputs()[0]
    print(f"Q: {question}")
    print(f"A: {answer}")
    print()

> Building 1 image...

> Building image rag-agent-lite for environment rag_pipeline

> Image localhost:30000/rag-agent-lite:3f1e6930f8960ab043f708426dfdea1c not found, building...

06:15:01.277585 WARNING   docker_builder.py:775 - Temporary directory: /var/folders/59/q0qx_fx10bxbxhpkyrcs79sr0000gn/T/tmpketqsymn

Run command: docker buildx build --builder flytex --tag localhost:30000/rag-agent-lite:3f1e6930f8960ab043f708426dfdea1c --platform linux/amd64,linux/arm64 --push /var/folders/59/q0qx_fx10bxbxhpkyrcs79sr0000gn/T/tmpketqsymn 


...

> Building 1 image...

> Building image rag-agent-lite for environment rag_pipeline

✓ Built image for environment rag_pipeline: localhost:30000/rag-agent-lite:3f1e6930f8960ab043f708426dfdea1c

Output()

Q: What is a TaskEnvironment in Flyte v2?
A: None



### Notes

- **Retrieval is in-process** (`@flyte.trace`), so each question runs in a single pod and still shows the retrieval step as a nested, traced action in the UI — no extra pod per call, nothing to OOM or queue on a single-node devbox.
- **Swapping in real semantic search**: replace the body of `retrieve_documents` with a vector-store query (e.g. ChromaDB/Weaviate/pgvector + an embedding model). On a real cluster, make it a separate `@env.task` so retrieval is isolated and parallelizable; size that task's memory for your embedding model and consider Union `ReusePolicy` to keep the model warm across calls.

## Scaling the pattern

This devbox example keeps everything in one lightweight pod. For production RAG:

- Move retrieval to its own `@env.task` backed by a vector store, sized for your embedding model, so it is isolated and horizontally scalable.
- On Union, use `ReusePolicy` to keep warm containers (and a loaded embedding model) around, eliminating per-call cold starts. `ReusePolicy` is a Union feature and is not supported on the local devbox.